## <a name="yolo">You Only Look Once (YOLO)</a>

### Backbone

#### YOLOv1中的网络结构
YOLOv1的网络结构灵感来自GoogLeNet，包括24个卷积层+2个全连接层。放弃了GoogLeNet的inception模块，而是使用1×1的下降层和3×3的卷积层，结构如下图所示。

![YOLOv1-architecture](./img/yolo-arc-1.png)

#### YOLO9000中的网络结构
YOLO9000中将DarkNet-19作为yolo_v2的网络结构，结构定义如下图所示。
![YOLOv2-architecture](./img/yolo-arc-2.png)

相比较YOLOv1，Darknet-19中的机构更加复杂，并且加入了Batch Normalization层，加速收敛。

#### YOLOv3中的网络结构
YOLOv3中将Darknet-53作为网络结构，结构如下图所示：
![YOLOv3-architecture](./img/yolo-arc-3.png)

相比较YOLOv2，YOLOv3的改进：

- Darknet-53加深了层结构，加深后有53层卷积层
- Darknet-53加入了残差结构，减少网络层加深对网络性能的影响
- Darknet-53用卷积层代替了池化层和全连接层，通过卷积层的步长来控制张量的尺寸

下面我们来对YOLOv3的网络结构进行详细的解析

In [ ]:
from functools import wraps

import numpy as np
import tensorflow as tf
from keras import backend as K
from keras.layers import Conv2D, Add, ZeroPadding2D, UpSampling2D, Concatenate, MaxPooling2D, Input
from keras.layers.advanced_activations import LeakyReLU
from keras.layers.normalization import BatchNormalization
from keras.models import Model
from keras.regularizers import l2

from src.yolo3.utils import compose

`DarknetConv2D`定义了卷积操作

In [ ]:
def DarknetConv2D(*args, **kwargs):
    """Wrapper to set Darknet parameters for Convolution2D."""
    darknet_conv_kwargs = {'kernel_regularizer': l2(5e-4)}
    darknet_conv_kwargs['padding'] = 'valid' if kwargs.get('strides')==(2,2) else 'same'
    darknet_conv_kwargs.update(kwargs)
    return Conv2D(*args, **darknet_conv_kwargs)

`DarknetConv2D_BN_Leaky`定义了一个卷积单位，一个卷积层，一个Batch Normalization层以及一个LeakyRELU激活函数。

> Batch Normalization 批归一化

在YOLO9000中提出在结构中加入BN层，旨在加深层结构的同时不会影响训练的效果。在层结构加深的同时，很可能会出现梯度消失等问题，所以加入BN层，进行批归一化。

> LeakyRELU 激活函数

ReLU是将所有的负值都设为零，相反，Leaky ReLU是给所有负值赋予一个非零斜率。
![DN](./img/leaky_relu.png)

In [ ]:
def DarknetConv2D_BN_Leaky(*args, **kwargs):
    """Darknet Convolution2D followed by BatchNormalization and LeakyReLU."""
    no_bias_kwargs = {'use_bias': False}
    no_bias_kwargs.update(kwargs)
    return compose(
        DarknetConv2D(*args, **no_bias_kwargs),
        BatchNormalization(),
        LeakyReLU(alpha=0.1))

`resblock_body` 是残差网络的基本模块，由零填充层和DarknetConv2D_BN_Leaky层组成，通过残差模块加深了网络结构。

![DN](./img/resblock_body.png)


In [ ]:
def resblock_body(x, num_filters, num_blocks):
    '''A series of resblocks starting with a downsampling Convolution2D'''
    # Darknet uses left and top padding instead of 'same' mode
    x = ZeroPadding2D(((1,0),(1,0)))(x)
    x = DarknetConv2D_BN_Leaky(num_filters, (3,3), strides=(2,2))(x)
    for i in range(num_blocks):
        y = compose(
                DarknetConv2D_BN_Leaky(num_filters//2, (1,1)),
                DarknetConv2D_BN_Leaky(num_filters, (3,3)))(x)
        x = Add()([x,y])
    return x

In [ ]:
def darknet_body(x):
    '''Darknent body having 52 Convolution2D layers'''
    x = DarknetConv2D_BN_Leaky(32, (3,3))(x)
    x = resblock_body(x, 64, 1)
    x = resblock_body(x, 128, 2)
    x = resblock_body(x, 256, 8)
    x = resblock_body(x, 512, 8)
    x = resblock_body(x, 1024, 4)
    return x

In [ ]:
def make_last_layers(x, num_filters, out_filters):
    '''6 Conv2D_BN_Leaky layers followed by a Conv2D_linear layer'''
    x = compose(
            DarknetConv2D_BN_Leaky(num_filters, (1,1)),
            DarknetConv2D_BN_Leaky(num_filters*2, (3,3)),
            DarknetConv2D_BN_Leaky(num_filters, (1,1)),
            DarknetConv2D_BN_Leaky(num_filters*2, (3,3)),
            DarknetConv2D_BN_Leaky(num_filters, (1,1)))(x)
    y = compose(
            DarknetConv2D_BN_Leaky(num_filters*2, (3,3)),
            DarknetConv2D(out_filters, (1,1)))(x)
    return x, y

YOLOv3网络有三个输出，_y1_，_y2_和_y3_。分别是三种不同尺寸的输出为13×13，26×26，52×52。这样网络将图片进行了不同尺寸的分割，以便于检测不同大小的目标。以y1为例，输出特征图为13×13×51，特征图的大小为13×13，可以看做是将输入图片分割成了13×13个部分，然后对每一个部分预测了51个值，3×（5+12）=51，y1预测**3个anchor** 的值，anchor来自之前介绍过的通过K-means计算得来，每个anchor预测**12个分类** （BDD100K类别为12），预测**5个值:w，h，x，y** ，**confidence（置信度）** 。
YOLOv3的网络结构（COCO数据集80个类别为例）如下图所示：

![full-arc](./img/full_arc.png)

YOLOv3的网络输出包括5个特征向量：w，h，x，y，confidence。
YOLO系列相比较于其他目标检测算法的区别之一是将预测值中的坐标变为预测偏移量，预测出b-box中心点相对于网格单元左上角的相对坐标。

![yolo-tensors](./img/yolo_tensors.png)

In [ ]:
inputs = Input(shape=(416, 416, 3))
num_anchors = 3
num_classes = 12
darknet = Model(inputs, darknet_body(inputs))
x, y1 = make_last_layers(darknet.output, 512, num_anchors*(num_classes+5))
x = compose(
        DarknetConv2D_BN_Leaky(256, (1,1)),
        UpSampling2D(2))(x)
x = Concatenate()([x,darknet.layers[152].output])
x, y2 = make_last_layers(x, 256, num_anchors*(num_classes+5))

x = compose(
        DarknetConv2D_BN_Leaky(128, (1,1)),
        UpSampling2D(2))(x)
x = Concatenate()([x,darknet.layers[92].output])
x, y3 = make_last_layers(x, 128, num_anchors*(num_classes+5))

网络的输出参数：

In [ ]:
print(y1)
print(y2)
print(y3)

YOLOv3的网络结构Darknet-53搭建完成。

## 数据准备

YOLO的目标检测使用数据集PASCAL VOC数据集以及COCO数据集，在综合案例中，我们使用BDD100K数据集，所以需要对数据集内容进行转换。
在BDD100K数据集中，标注信息类别为12种：
`['car', 'bus', 'person', 'bike', 'truck', 'motor', 'train', 'rider', 'traffic sign', 'traffic light', 'drivable area','lane']`

转换包括将标注信息提取出来，只保留：目标类别，目标坐标，图片信息内容。
图片准备过程在`bdd_annotations.py`中

## BDD100K Anchor计算

YOLO中，没有人工挑选Anchor，而是使用K-means算法挑选先验框。好的先验框可以使训练的效果更好，所以先验框的挑选十分重要。
##### K-means 算法

计算过程：

1. 随机选取k个中心点

2. 遍历所有数据，将每个数据划分到最近的中心点中

3. 计算每个聚类的平均值，并作为新的中心点

4. 重复步骤2，3，直到这k个中线点不再变化，或执行了足够多的迭代

在普遍的K-means算法中，计算聚类的距离平均值时使用欧几里得距离进行计算。在目标检测中，想要IOU的值最大，而且欧式距离计算结果出现了先验框尺寸较大的问题，所以计算平均IOU的值代替计算欧氏距离。参考表格，可以看到在聚类个数为5时，平均IOU的值，IOU计算方式要比欧式距离更高。
YOLO中平均IOU计算方式：


$$d(box,centroid) = 1- IOU(box, centroid)$$

IOU：预测值边框与真值ground truth边框的交集和并集的比值，能够表现出预测值的偏离程度。

在YOLO的anchor计算中，定义聚类个数为9。参考表格：
![yolo-chart](./img/yolo-chart.png)

_ClusterSSE表示通过传统的欧氏距离计算得出的先验框，ClusterIOU表示通过YOLO中定义的计算方式得出的先验框。Anchor Box表示人工挑选的先验框。_

可以看到，K-means计算的先验框在聚类个数为5的时候average IOU接近人工挑选时9个Anchorbox的情况，当K-means计算的先验框聚类个数为9时远超过了人工挑选，所以K-means效果比人工挑选更好。

##### BDD100K的K-means
运行`kmeans.py`，将生成BDD100K的anchor box。 

一共生成9组anchor：
`11,13, 20,18, 25,42, 42,25, 72,58, 114,4, 191,119, 317,27, 495,237`
可以将其分成三组：`11,13, 20,18, 25,42` ,`42,25, 72,58, 114,4` ,`191,119, 317,27, 495,237` ,
在检测时，对感受野较大，尺寸较小的特征图应用较大的anchor，以检测较大的目标，而对尺寸较大，对应感受野较小的特征图应用较小的anchor，检测更小的目标，这样可以保证模型在检测不同大小目标时的效果。

## YOLOv3的图片测试
在YOLOv3，COCO预训练模型的基础上，进行YOLOv3对图片的目标检测测试。

In [ ]:
from PIL import Image
import numpy as np
from keras.layers import Input
from src.yolo3.utils import letterbox_image
from src.yolo3.model import yolo_body

打开图片，并将图片转为nparray
- IMAGE_PATH = 测试图片路径

In [ ]:
IMAGE_PATH = './data/Tesla-Autopilot-sees-danger-ahead.png'
image = Image.open(IMAGE_PATH)
image_ori = np.array(image)
print("测试图片大小为："+str(image_ori.shape))

图像预处理：
- 对输入图像的尺寸进行调整，调整为宽高都是32的整数倍数
- 对输入图像进行标准化处理，提高训练速度
- 输入图像加入维度，进行批处理

In [ ]:
new_image_size = (image.width - (image.width % 32), image.height - (image.height % 32))
boxed_image = letterbox_image(image, new_image_size)
image_data = np.array(boxed_image, dtype='float32')
image_data /= 255.
image_data = np.expand_dims(image_data, 0)
print("预处理之后的图片大小为："+str(image_data.shape))

读取anchor文件，存储在变量`anchors`中。Anchor中存储了BDD100K数据集中的数据，经过kmeans计算后的数值。

In [ ]:
anchor_path = "./data/yolov3_anchors.txt"
with open(anchor_path) as f:
    anchors = f.readline()
anchors = [float(x) for x in anchors.split(',')]
anchors = np.array(anchors).reshape(-1, 2)

构建YOLO模型结构，构建需要如下数值：
- (?,?,3) 形状的输入变量
- anchor，一共有9个anchor，一共有3层需要anchor，所以需要的anchor数量为`len(anchors)//3`
- 识别的类型数量，在这里使用的预训练模型针对COCO数据集，所以类型数量为80

In [ ]:
yolo_model = yolo_body(Input(shape=(None,None,3)), len(anchors)//3, 80)
weights_path = "./data/yolov3.h5"
yolo_model.load_weights(weights_path)

读取模型之后，可以看到模型的输入和输出，对照模型结构进行理解。

In [ ]:
yolo_model.input

In [ ]:
yolo_model.output

输入模型结果进行评估，并进行边界框过滤。

In [ ]:
from src.yolo3.model import yolo_eval
input_image_shape = K.placeholder(shape=(2, ))
boxes, scores, classes = yolo_eval(
    yolo_model.output, 
    anchors,
    80, 
    input_image_shape,
    score_threshold=0.3, 
    iou_threshold=0.45)

In [ ]:
print(boxes)
print(scores)
print(classes)

将数据传入模型进行计算

In [ ]:
import keras.backend as K
sess = K.get_session()
out_boxes, out_scores, out_classes = sess.run(
    [boxes, scores, classes],
    feed_dict={
        yolo_model.input: image_data,
        input_image_shape: [image.size[1], image.size[0]],
        K.learning_phase(): 0
    })
sess.close()

In [ ]:
print(out_boxes)
print(out_scores)
print(out_classes)

将结果进行绘制

In [ ]:
from PIL import Image, ImageFont, ImageDraw

# font = ImageFont.truetype(font='font/FiraMono-Medium.otf',
#                     size=np.floor(3e-2 * image.size[1] + 0.5).astype('int32'))

thickness = (image.size[0] + image.size[1]) // 300

for i, c in reversed(list(enumerate(out_classes))):
    predicted_class = c
    box = out_boxes[i]
    score = out_scores[i]

    label = '{} {:.2f}'.format(predicted_class, score)
    draw = ImageDraw.Draw(image)
#     label_size = draw.textsize(label, font)

    top, left, bottom, right = box
    top = max(0, np.floor(top + 0.5).astype('int32'))
    left = max(0, np.floor(left + 0.5).astype('int32'))
    bottom = min(image.size[1], np.floor(bottom + 0.5).astype('int32'))
    right = min(image.size[0], np.floor(right + 0.5).astype('int32'))
    print(label, (left, top), (right, bottom))

    if top - label_size[1] >= 0:
        text_origin = np.array([left, top - label_size[1]])
    else:
        text_origin = np.array([left, top + 1])

    for i in range(thickness):
        draw.rectangle(
            [left + i, top + i, right - i, bottom - i],
            outline=225)
    draw.rectangle(
        [tuple(text_origin), tuple(text_origin + label_size)],
        fill=225)
#     draw.text(text_origin, label, fill=(0, 0, 0), font=font)
    del draw

In [ ]:
image

## YOLOv3 在BDD100K上的训练

利用BDD100K数据集对YOLOv3进行训练。

引入相关的包

In [ ]:
import keras.backend as K
import numpy as np
from keras.layers import Input, Lambda
from keras.models import Model
from keras.optimizers import Adam

from src.yolo3.train import data_generator_wrapper
from src.yolo3.model import preprocess_true_boxes, yolo_body, yolo_loss

定义参数

In [ ]:
from src.yolo3.train import get_classes, get_anchors
# 数据集类别存储在bdd_classes.txt中
classes_path = './model_data/bdd_classes.txt'
# 数据集anchor储存在bdd_anchors.txt中
anchors_path = './bdd_anchors.txt'
# 数据集的标注信息储存在bdd100k_train.txt中
annotation_path = './data/bdd100k_train.txt'
# 预训练模型yolo.h5
weights_path = "./model_data/yolo.h5"
# 读取类别和anchor文件
classes = get_classes(classes_path)
anchors = get_anchors(anchors_path)
# 类别和anchor长度
num_classes = len(classes)
num_anchors = len(anchors)
# 输入图片需要为13的倍数，方便grid cell的划分
input_shape = (416,416)

In [ ]:
# 清空session
K.clear_session()
# 定义输入数据的格式
image_input = Input(shape=(None, None, 3))
h, w = input_shape

# 数据有三种格式（宽*高）：13*13, 26*26, 52*52 
y_true = [Input(shape=(h//{0:32, 1:16, 2:8}[l], w//{0:32, 1:16, 2:8}[l], num_anchors//3, num_classes+5)) for l in range(3)]
# num_anchor //3 每个特征图只对应3个anchor
model_body = yolo_body(image_input, num_anchors//3, num_classes)
# 载入权重
model_body.load_weights(weights_path, by_name=True, skip_mismatch=True)
# 定义损失函数
model_loss = Lambda(yolo_loss, output_shape=(1,), name='yolo_loss',
    arguments={'anchors': anchors, 'num_classes': num_classes, 'ignore_thresh': 0.5})(
    [*model_body.output, *y_true])
# 构建模型
model = Model([model_body.input, *y_true], model_loss)

定义优化方法

In [ ]:
from keras.callbacks import ReduceLROnPlateau, EarlyStopping
# 降低学习率
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, verbose=1)
# 早停法避免过拟合
early_stopping = EarlyStopping(monitor='val_loss', min_delta=0, patience=10, verbose=1)

In [ ]:
# 验证集占训练集比例
val_split = 0.1
# 读取标注信息
with open(annotation_path) as f:
    lines = f.readlines()
np.random.seed(10101)
np.random.shuffle(lines)
np.random.seed(None)
num_val = int(len(lines)*val_split)
num_train = len(lines) - num_val

In [ ]:
batch_size = 4 
model.compile(optimizer=Adam(lr=1e-4), loss={'yolo_loss': lambda y_true, y_pred: y_pred}) 

开始模型训练，将训练后的模型通过save_weights保存权重信息

In [ ]:
print('Train on {} samples, val on {} samples, with batch size {}.'.format(num_train, num_val, batch_size))
model.fit_generator(data_generator_wrapper(lines[:num_train], batch_size, input_shape, anchors, num_classes),
    steps_per_epoch=max(1, num_train//batch_size),
    validation_data=data_generator_wrapper(lines[num_train:], batch_size, input_shape, anchors, num_classes),
    validation_steps=max(1, num_val//batch_size),
    epochs=100,
    initial_epoch=0,
    callbacks=[reduce_lr, early_stopping])
model.save_weights(log_dir + 'trained_weights_final.h5')